# 01 · Volatility Forecasting — ML vs GARCH Baseline

**Project 8 / 10 — ML Risk Estimation & Forecasting**  
Random Forest and XGBoost trained against a GARCH-proxy baseline.  
SR 11-7: challenger model comparison with documented RMSE benchmarks.

> Author: Niraj Neupane | github.com/nirajneupane17

In [ ]:
import sys; sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings("ignore")
from sklearn.metrics import mean_squared_error, mean_absolute_error
from feature_engineering import build_vol_features, train_test_split_temporal
from ml_models import VolatilityForecaster
print("Libraries loaded")

In [ ]:
# Load SPY returns
rets = pd.read_csv("../data/returns.csv", index_col="Date", parse_dates=True)
spy  = rets["SPY"].dropna()
print(f"Date range : {spy.index[0].date()} → {spy.index[-1].date()}")
print(f"Ann. vol   : {spy.std()*np.sqrt(252)*100:.1f}%")
print(f"Ann. return: {spy.mean()*252*100:.1f}%")
spy.describe()

In [ ]:
# Build features
feat = build_vol_features(spy, windows=[5,10,21,63], target_horizon=21)
X_cols = ["vol_5d","vol_10d","vol_21d","vol_63d","skewness_21d","kurtosis_21d",
          "return_sq","abs_return","vol_ratio","momentum_5d","momentum_21d"]
X_train, X_test, y_train, y_test = train_test_split_temporal(feat, test_frac=0.20, target_col="vol_target")
X_train_f = X_train[X_cols]; X_test_f = X_test[X_cols]
print(f"Train: {X_train_f.shape}  Test: {X_test_f.shape}")

In [ ]:
# Fit VolatilityForecaster (RF + XGBoost)
vf = VolatilityForecaster()
vf.fit(X_train_f, y_train)
results = vf.evaluate(X_test_f, y_test)
pd.DataFrame(results).T

In [ ]:
# GARCH proxy RMSE for reference
garch_pred = X_test_f["vol_21d"].values
garch_rmse = np.sqrt(mean_squared_error(y_test.values, garch_pred))
rf_rmse    = results["rf"]["rmse"]
xgb_rmse   = results["xgb"]["rmse"]
print(f"GARCH RMSE  : {garch_rmse:.4f}  (benchmark)")
print(f"RF RMSE     : {rf_rmse:.4f}  ({(garch_rmse-rf_rmse)/garch_rmse*100:+.1f}% vs GARCH)")
print(f"XGBoost RMSE: {xgb_rmse:.4f}  ({(garch_rmse-xgb_rmse)/garch_rmse*100:+.1f}% vs GARCH)")
print(f"Ensemble    : {results['ensemble']['rmse']:.4f}")

In [ ]:
# Visualise — last 252 trading days
img = plt.imread("../results/01_volatility_forecast.png")
fig, ax = plt.subplots(figsize=(15,7)); ax.imshow(img); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
# Feature importance
fi = pd.Series(vf.rf.feature_importances_, index=X_cols).sort_values(ascending=False)
print("Top 5 RF features:"); print(fi.head(5).to_string())